In [ ]:
import os, glob, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Config
# -----------------------------
RESULT_DIR = "./result"
OUT_DIR = "./analysis_out/ch5_4/figures"
TABLE_DIR = "./analysis_out/ch5_4/tables"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

# If your SBM uses k=3 communities (as in your data generation code), keep k=3.
K_COMMUNITIES = 3
LOG2K = math.log2(K_COMMUNITIES)

# Binning settings (tune if needed)
N_BINS = 6
MIN_BIN_N = 30   # drop bins with too few trials to avoid noisy thresholds

# Event label normalization (your UI uses "Continue" instead of "Stable")
EVENT_CANON = {
    "Stable": "Continue",
    "Continue": "Continue",
    "SizeChange": "SizeChange",
    "Merge": "Merge",
    "Split": "Split",
}

# -----------------------------
# Robust loader
# -----------------------------
def load_all_trials(result_dir=RESULT_DIR):
    files = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    if not files:
        raise FileNotFoundError(f"No json files found in {result_dir}")

    rows = []
    bad_files = 0

    for fp in files:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                obj = json.load(f)
        except Exception as e:
            print(f"[WARN] Failed to read {fp}: {e}")
            bad_files += 1
            continue

        # Your result file is typically a list of trials (len=72)
        if isinstance(obj, dict) and "trials" in obj:
            trials = obj["trials"]
        elif isinstance(obj, list):
            trials = obj
        else:
            print(f"[WARN] Unrecognized json structure in {fp} (type={type(obj)}). Skip.")
            bad_files += 1
            continue

        for t in trials:
            if not isinstance(t, dict):
                continue

            # ---- key: h_alias in your files ----
            halias_raw = (
                t.get("h_alias", None) or
                t.get("H_alias", None) or
                t.get("halias", None) or
                t.get("halias_raw", None)
            )

            # if still missing, keep None (we will drop later)
            event = t.get("ground_truth", None) or t.get("event_type", None) or t.get("event", None)
            event = EVENT_CANON.get(event, event)

            row = {
                "participant": t.get("subject_id", t.get("participant_id", "unknown")),
                "trial_index": t.get("trial_index", t.get("id", None)),
                "W": t.get("W", None),
                "dataset": t.get("dataset_tag", t.get("dataset", None)),
                "event": event,
                "correct": t.get("correct", None),
                "answer": t.get("answer", None),
                "confidence": t.get("confidence", None),
                "rt_ms": t.get("duration_ms", t.get("rt_ms", None)),
                "toggles": t.get("space_toggle_count", t.get("toggles", None)),
                "mouse_entropy": t.get("mouse_entropy", None),
                "halias_raw": halias_raw,
            }
            rows.append(row)

    df = pd.DataFrame(rows)

    # Ensure expected columns exist even if df is empty
    for col in ["participant","trial_index","W","dataset","event","correct","halias_raw"]:
        if col not in df.columns:
            df[col] = np.nan

    print(f"[load] files={len(files)}, bad_files={bad_files}, rows={len(df)}")
    return df

def clean_and_derive(df: pd.DataFrame) -> pd.DataFrame:
    # drop missing key variables
    before = len(df)
    df = df.dropna(subset=["halias_raw", "correct"]).copy()
    after = len(df)
    print(f"[clean] drop missing halias/correct: {before} -> {after}")

    # type casting
    df["correct"] = df["correct"].astype(int)

    # derive normalized aliasing entropy
    df["halias_norm"] = df["halias_raw"] / LOG2K

    # optional: remove Continue if you want thresholds only on “non-trivial events”
    # df = df[df["event"] != "Continue"].copy()

    return df

# -----------------------------
# Binning + CI helpers
# -----------------------------
def bin_stats(df, x="halias_norm", y="correct", n_bins=N_BINS):
    """
    Equal-width bins on x in [min,max].
    Return per-bin mean(y), 95% CI (normal approx), count, bin_center.
    """
    xmin, xmax = df[x].min(), df[x].max()
    if xmin == xmax:
        raise ValueError("x has zero range; cannot bin.")

    edges = np.linspace(xmin, xmax, n_bins + 1)
    df = df.copy()
    df["bin"] = pd.cut(df[x], bins=edges, include_lowest=True)

    out = []
    for b, g in df.groupby("bin", observed=True):
        n = len(g)
        if n == 0:
            continue
        m = g[y].mean()

        # 95% CI for mean of Bernoulli: normal approx on proportion
        # (ok for moderate n; we also drop tiny bins)
        se = math.sqrt(max(m * (1 - m), 1e-12) / n)
        ci = 1.96 * se

        left, right = b.left, b.right
        center = (left + right) / 2
        out.append({"bin": str(b), "left": left, "right": right, "center": center,
                    "n": n, "mean": m, "ci95": ci})
    out = pd.DataFrame(out).sort_values("center")

    # drop small bins to stabilize thresholding
    out = out[out["n"] >= MIN_BIN_N].copy()
    return out, edges

# -----------------------------
# Simple change-point on binned means (two-segment fit)
# -----------------------------
def find_threshold_from_binned(binned: pd.DataFrame):
    """
    Choose a split index s (between bins) minimizing SSE of two horizontal means:
      bins <= s use mean1, bins > s use mean2.
    Return best threshold x* (midpoint between bins) and stats.
    """
    x = binned["center"].values
    y = binned["mean"].values
    n = len(binned)
    if n < 4:
        return None  # not enough bins

    best = None
    for s in range(1, n-1):
        y1 = y[:s+1]
        y2 = y[s+1:]
        m1 = y1.mean()
        m2 = y2.mean()
        sse = ((y1 - m1)**2).sum() + ((y2 - m2)**2).sum()
        if (best is None) or (sse < best["sse"]):
            # threshold located between bin s and s+1
            thr = (x[s] + x[s+1]) / 2
            best = {"split_idx": s, "thr": thr, "mean_low": m1, "mean_high": m2, "sse": float(sse)}
    return best

def summarize_pre_post(df, thr):
    pre = df[df["halias_norm"] <= thr]
    post = df[df["halias_norm"] > thr]
    def summ(g):
        n = len(g)
        acc = g["correct"].mean() if n else np.nan
        return n, acc
    n1, a1 = summ(pre)
    n2, a2 = summ(post)
    return pd.DataFrame([{"group":"<=thr", "n":n1, "accuracy":a1},
                         {"group":">thr", "n":n2, "accuracy":a2}])

# -----------------------------
# Main (5.4)
# -----------------------------
def main():
    df = load_all_trials(RESULT_DIR)
    df = clean_and_derive(df)

    # ---- 1) overall binned accuracy curve ----
    b_overall, edges = bin_stats(df, x="halias_norm", y="correct", n_bins=N_BINS)
    b_overall.to_csv(os.path.join(TABLE_DIR, "table_5_4_binned_accuracy_overall.csv"), index=False)

    # ---- 2) threshold (change-point) on binned means ----
    thr_info = find_threshold_from_binned(b_overall)
    if thr_info is None:
        print("[thr] Not enough stable bins to estimate a threshold.")
        thr = None
    else:
        thr = thr_info["thr"]
        print(f"[thr] estimated thr={thr:.3f}, mean_low={thr_info['mean_low']:.3f}, mean_high={thr_info['mean_high']:.3f}")

        with open(os.path.join(TABLE_DIR, "table_5_4_threshold.txt"), "w", encoding="utf-8") as f:
            f.write(f"Estimated threshold (on halias_norm): {thr:.6f}\n")
            f.write(f"Split bin idx: {thr_info['split_idx']}\n")
            f.write(f"Mean accuracy (<=thr): {thr_info['mean_low']:.6f}\n")
            f.write(f"Mean accuracy (>thr): {thr_info['mean_high']:.6f}\n")
            f.write(f"SSE: {thr_info['sse']:.6f}\n")

        prepost = summarize_pre_post(df, thr)
        prepost.to_csv(os.path.join(TABLE_DIR, "table_5_4_pre_post_threshold.csv"), index=False)

    # ---- Plot: accuracy vs halias_norm (binned mean ±95% CI) ----
    plt.figure()
    plt.plot(b_overall["center"], b_overall["mean"], marker="o")
    plt.fill_between(b_overall["center"],
                     b_overall["mean"] - b_overall["ci95"],
                     b_overall["mean"] + b_overall["ci95"],
                     alpha=0.2)
    if thr is not None:
        plt.axvline(thr, linestyle="--")
    plt.ylim(0, 1.0)
    plt.xlabel(r"Normalized aliasing entropy $\tilde{H}_{alias}$")
    plt.ylabel("Accuracy")
    plt.title(r"Accuracy vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig_5_4_accuracy_threshold_overall.png"), dpi=200)

    # ---- Optional: threshold per event (excluding Continue often makes sense) ----
    # You can uncomment if you want event-specific thresholds.
    """
    for ev, g in df[df["event"]!="Continue"].groupby("event"):
        b_ev, _ = bin_stats(g, x="halias_norm", y="correct", n_bins=N_BINS)
        if len(b_ev) < 4:
            continue
        info = find_threshold_from_binned(b_ev)
        if info is None:
            continue
        thr_ev = info["thr"]

        plt.figure()
        plt.plot(b_ev["center"], b_ev["mean"], marker="o")
        plt.fill_between(b_ev["center"], b_ev["mean"]-b_ev["ci95"], b_ev["mean"]+b_ev["ci95"], alpha=0.2)
        plt.axvline(thr_ev, linestyle="--")
        plt.ylim(0,1.0)
        plt.xlabel(r"Normalized aliasing entropy $\tilde{H}_{alias}$")
        plt.ylabel("Accuracy")
        plt.title(fr"{ev}: Accuracy vs $\tilde{{H}}_{{alias}}$ (thr={thr_ev:.2f})")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"fig_5_4_threshold_{ev}.png"), dpi=200)
    """

    print(f"[done] figs -> {OUT_DIR}")
    print(f"[done] tables -> {TABLE_DIR}")

if __name__ == "__main__":
    main()

In [ ]:
import os, json, glob
import pandas as pd

# -----------------------------
# 1️⃣ 读取所有 result 文件
# -----------------------------
def load_all_results(result_dir="./result"):
    rows = []
    paths = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    print("Files found:", len(paths))

    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # ✅ 你的文件是 list
        if not isinstance(data, list):
            print("Unexpected structure:", type(data), "in", path)
            continue

        pid = os.path.basename(path)

        for t in data:
            if not isinstance(t, dict):
                continue

            rows.append({
                "participant": pid,
                "event": t.get("event_type") or t.get("event"),
                "W": t.get("W"),
                "dataset": t.get("dataset_tag") or t.get("dataset"),
                "halias_norm": t.get("h_alias") or t.get("halias_norm"),
                "correct": t.get("correct"),
            })

    df = pd.DataFrame(rows)
    return df

df = load_all_results("./result")

print("Raw rows:", len(df))
print("Columns:", df.columns.tolist())
print(df.head())

# -----------------------------
# 2️⃣ 清洗
# -----------------------------
df = df.dropna(subset=["halias_norm", "correct", "event", "W"]).copy()

df["halias_norm"] = pd.to_numeric(df["halias_norm"], errors="coerce")
df["correct"] = pd.to_numeric(df["correct"], errors="coerce")
df["W"] = pd.to_numeric(df["W"], errors="coerce")

df = df.dropna(subset=["halias_norm", "correct", "W"])

print("\nAfter cleaning:", len(df))

# -----------------------------
# 3️⃣ 看高 aliasing 样本
# -----------------------------
thr = 0.575
high = df[df["halias_norm"] > thr]

print("\nHigh aliasing trials:", len(high))

print("\nEvent distribution:")
print(high["event"].value_counts())

print("\nW distribution:")
print(high["W"].value_counts().sort_index())

print("\nDataset distribution:")
print(high["dataset"].value_counts())

print("\nParticipants involved:", high["participant"].nunique())

In [ ]:
import os, json, glob
import pandas as pd
import numpy as np
import math

def load_all_results(result_dir="./result"):
    rows = []
    paths = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    print("Files found:", len(paths))

    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            print("Unexpected structure:", type(data), "in", path)
            continue

        pid = os.path.basename(path)

        for t in data:
            if not isinstance(t, dict):
                continue

            # ✅ 用 ground_truth 才是事件真值
            ev = t.get("ground_truth") or t.get("event") or t.get("event_type")

            # ✅ 你文件里是 h_alias（未必归一化）
            h = t.get("h_alias")
            # 若你的文件也有 h_alias_norm，则优先用
            h_norm = t.get("h_alias_norm") if ("h_alias_norm" in t) else None

            rows.append({
                "participant": pid,
                "event": ev,
                "W": t.get("W"),
                "dataset": t.get("dataset_tag") or t.get("dataset"),
                "h_alias_raw": h,
                "h_alias_norm": h_norm,
                "correct": t.get("correct"),
            })

    df = pd.DataFrame(rows)
    return df

df = load_all_results("./result")

print("Raw rows:", len(df))
print(df[["event","W","dataset","h_alias_raw","h_alias_norm","correct"]].head())

# --------- cleaning ----------
# event 可以先不 drop（但如果要做按事件分析，就必须有）
df["correct"] = pd.to_numeric(df["correct"], errors="coerce")
df["W"] = pd.to_numeric(df["W"], errors="coerce")
df["h_alias_raw"] = pd.to_numeric(df["h_alias_raw"], errors="coerce")

# ✅ 归一化：你的 k=3
LOG2K = math.log2(3)
df["halias_norm"] = df["h_alias_raw"] / LOG2K

# 只 drop 必要字段
df_clean = df.dropna(subset=["halias_norm", "correct", "W"]).copy()

print("\nAfter cleaning:", len(df_clean))
print("Missing event rate:", df_clean["event"].isna().mean())

# --------- high aliasing inspection ----------
thr = 0.575
high = df_clean[df_clean["halias_norm"] > thr]

print("\nHigh aliasing trials:", len(high))

print("\nEvent distribution (high) [including NaN if any]:")
print(high["event"].value_counts(dropna=False))

print("\nW distribution (high):")
print(high["W"].value_counts(dropna=False).sort_index())

print("\nDataset distribution (high):")
print(high["dataset"].value_counts(dropna=False))

print("\nParticipants involved (high):", high["participant"].nunique())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 假设 df_clean 已存在（上一段代码生成）
df = df_clean.copy()

# 只保留 Merge
df_m = df[df["event"] == "Merge"].copy()

print("Merge trials:", len(df_m))
print("Participants:", df_m["participant"].nunique())

# ---------------------------
# 1️⃣ Merge: accuracy vs halias_norm (binned)
# ---------------------------

n_bins = 8
df_m["h_bin"] = pd.cut(df_m["halias_norm"], bins=n_bins)

grouped = df_m.groupby("h_bin").agg(
    mean_h=("halias_norm", "mean"),
    acc=("correct", "mean"),
    n=("correct", "count")
).reset_index()

plt.figure()
plt.plot(grouped["mean_h"], grouped["acc"], marker="o")
plt.xlabel("Normalized Aliasing Entropy (Merge only)")
plt.ylabel("Accuracy")
plt.title("Merge: Accuracy vs Aliasing Entropy")
plt.ylim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.savefig("fig_5_4_merge_accuracy_vs_halias.png", dpi=300)
plt.show()

print("\nBinned summary (Merge):")
print(grouped)


# ---------------------------
# 2️⃣ Merge: accuracy vs W
# ---------------------------

acc_by_W = df_m.groupby("W")["correct"].mean().reset_index()

plt.figure()
plt.plot(acc_by_W["W"], acc_by_W["correct"], marker="o")
plt.xlabel("Aggregation Window Size (W)")
plt.ylabel("Accuracy (Merge)")
plt.title("Merge: Accuracy vs Window Size")
plt.ylim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.savefig("fig_5_4_merge_accuracy_vs_W.png", dpi=300)
plt.show()

print("\nAccuracy by W (Merge):")
print(acc_by_W)


# ---------------------------
# 3️⃣ Merge: halias_norm vs W
# ---------------------------

h_by_W = df_m.groupby("W")["halias_norm"].mean().reset_index()

plt.figure()
plt.plot(h_by_W["W"], h_by_W["halias_norm"], marker="o")
plt.xlabel("Aggregation Window Size (W)")
plt.ylabel("Mean Aliasing Entropy (Merge)")
plt.title("Merge: Aliasing Entropy vs Window Size")
plt.grid(True)
plt.tight_layout()
plt.savefig("fig_5_4_merge_halias_vs_W.png", dpi=300)
plt.show()

print("\nAliasing by W (Merge):")
print(h_by_W)